## Load sub

The videos are in the order of Experiment_id, so not in the order of presentation. This means the first video is the same for each participant.

In [1]:
import pickle
import numpy as np
import gc  # Garbage collector

!pip install mrmr-selection
!pip install ordpy

In [2]:
participants = 32
subjects = {'data': [], 'labels': []}
prefix_path = '/kaggle/input/deap-dataset/deap-dataset/'

# Διαβάζουμε τα δεδομένα κάθε συμμετέχοντα
for i in range(1, participants + 1):
    file_name = prefix_path + f"data_preprocessed_python/s{'0' if i < 10 else ''}{i}.dat"
    with open(file_name, 'rb') as file:
        subject = pickle.load(file, encoding='latin1')
        for key in subjects:
            subjects[key].append(subject[key])

        del subject

# Ενοποίηση δεδομένων από όλους τους συμμετέχοντες
for key in subjects:
    subjects[key] = np.concatenate(subjects[key], axis=0)

print(subjects['data'].shape)
print(subjects['labels'].shape)

(1280, 40, 8064)
(1280, 4)


In [3]:
# Κρατάμε μόνο Valence και Arousal
subjects['labels'] = subjects['labels'][:, :2]

# Δυαδική κατηγοριοποίηση με threshold = 5
subjects['labels'][:, 0] = (subjects['labels'][:, 0] >= 5).astype(int)
subjects['labels'][:, 1] = (subjects['labels'][:, 1] >= 5).astype(int)

print(set(subjects['labels'][:, 0]))
print(set(subjects['labels'][:, 1]))

{0.0, 1.0}
{0.0, 1.0}


## Split Train Test Dataset

In [4]:
from sklearn.model_selection import train_test_split

test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(
    subjects['data'], subjects['labels'], test_size=test_size, random_state=random_state
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

del subjects
gc.collect()  # Καθαρισμός μνήμης

(1024, 40, 8064) (256, 40, 8064) (1024, 2) (256, 2)


0

## Feature extraction


### Wavelet transform

In [5]:
# Wavelet transform
import pywt
from tqdm import tqdm

# Συχνότητα δειγματοληψίας
fs = 128

# Επιλογή wavelet και scales
scales = np.arange(1, 129)  # Ví dụ: 129 giá trị scale
wavelet = 'morl'

def CWT_for_sample(data):
    cwt_output = []
    for i in tqdm(range(data.shape[0])):  # Για κάθε δείγμα
        cwt_channels = []
        for j in range(data.shape[1]):  # Για κάθε κανάλι EEG
            coeffs, freqs = pywt.cwt(data[i, j], scales, wavelet) 
            energy_scales = np.sum(np.abs(coeffs)**2, axis=1)  # Υπολογισμός ενέργειας ανά scale
            cwt_channels.append(energy_scales)
        cwt_output.append(cwt_channels)
    
    cwt_output = np.array(cwt_output)  

    return cwt_output
    
# cwt_train = CWT_for_sample(X_train) 
# cwt_test = CWT_for_sample(X_test) 

# print(cwt_train.shape, cwt_test.shape)

# # Tên file lưu
# prefix_output_path = '/kaggle/working/'
# filename = "cwt_data.pkl"

# # Lưu dữ liệu
# with open(filename, 'wb') as file:
#     pickle.dump({'train': cwt_train, 'test': cwt_test}, file)


In [6]:
# ===============================
# Φόρτωση αποθηκευμένων CWT features
# ===============================
prefix_path = '/kaggle/input/cwt-transform-deap/'
filename = "cwt_data.pkl"

with open(prefix_path + filename, 'rb') as file:
    data = pickle.load(file)

cwt_train_loaded = data['train']
cwt_test_loaded = data['test']

print("Kích thước cwt_train:", cwt_train_loaded.shape)
print("Kích thước cwt_test:", cwt_test_loaded.shape)


Kích thước cwt_train: (1024, 40, 128)
Kích thước cwt_test: (256, 40, 128)


### Nonlinear Feature Analyses: Permutation Entropy

In [7]:
# ===============================
# Feature Extraction – Permutation Entropy
# ===============================

import ordpy

def split_array(arr, num_epochs):
    """
    Διαχωρισμός σήματος σε epochs
    """
    len_arr = len(arr)
    len_epochs = round(len_arr / num_epochs)
    
    splits_arr = [arr[i * len_epochs: len_arr if i + 1 == num_epochs else (i + 1) * len_epochs] for i in range(num_epochs)]

    return splits_arr

def permutation_entropy_eeg(data, num_epochs=8):
    B, C, T = data.shape
    pe = np.zeros((B, C, num_epochs))
    
    for b in tqdm(range(B)):
        for c in range(C):
            splits_arr = split_array(data[b, c], num_epochs)
            for i in range(num_epochs):
                pe[b, c, i] = ordpy.permutation_entropy(splits_arr[i])

    return pe

# pe_train = permutation_entropy_eeg(X_train)#1024x40
# pe_test = permutation_entropy_eeg(X_test)#256x40

# # Tên file lưu
# prefix_output_path = '/kaggle/working/'
# filename = "pe_data.pkl"

# # Lưu dữ liệu
# with open(filename, 'wb') as file:
#    pickle.dump({'train': pe_train, 'test': pe_test}, file)

# print(f"Dữ liệu đã được lưu vào file {filename}")

In [8]:
prefix_path = '/kaggle/input/pe-transform-deap/'
filename = "pe_data.pkl"

with open(prefix_path + filename, 'rb') as file:
    data = pickle.load(file)

pe_train_loaded = data['train']
pe_test_loaded = data['test']

print("PE Train shape:", pe_train_loaded.shape)
print("PE Test shape:", pe_test_loaded.shape)

PE Train shape: (1024, 40, 8)
PE Test shape: (256, 40, 8)


## Feature Selection

In [9]:
# ===============================
# Feature Reduction – PCA
# ===============================

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def pca_data(data, k):
    """
    Εφαρμογή PCA σε EEG δεδομένα
    """
    # Flatten
    n_samples, n_channels, n_features = data.shape
    data_reshaped = data.reshape(n_samples * n_channels, n_features)
    
     # Κανονικοποίηση
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_reshaped)
    
    # PCA
    pca = PCA(n_components=k)
    data_pca = pca.fit_transform(data_scaled)
    
    # Reshape πίσω
    data_pca_reshaped = data_pca.reshape(n_samples, n_channels, k)
    
    return data_pca_reshaped

# pca_train = pca_data(X_train, k=128)
# pca_test = pca_data(X_test, k=128)

# # Tên file lưu
# prefix_output_path = '/kaggle/working/'
# filename = "pca_data.pkl"

# # Lưu dữ liệu
# with open(filename, 'wb') as file:
#    pickle.dump({'train': pca_train, 'test': pca_test}, file)

# print(f"Dữ liệu đã được lưu vào file {filename}")


In [10]:
# ===============================
# Φόρτωση αποθηκευμένων PCA features
# ===============================

prefix_path = '/kaggle/input/pca-transform-deap/'
filename = "pca_data.pkl"

with open(prefix_path + filename, 'rb') as file:
    data = pickle.load(file)

pca_train_loaded = data['train']
pca_test_loaded = data['test']

print("PCA Train shape:", pca_train_loaded.shape)
print("PCA Test shape:", pca_test_loaded.shape)


PCA Train shape: (1024, 40, 128)
PCA Test shape: (256, 40, 128)


In [11]:
# ===============================
# Feature Selection – mRMR
# ===============================

from mrmr import mrmr_classif
import pandas as pd

def feature_selector(num_features, X, y, classif_obj): 
    flatted_X = X.reshape(-1, X.shape[-1])

    flatted_X = pd.DataFrame(flatted_X)

    flatted_y_new = y
    if len(y.shape) == 2:
        y_new = np.repeat(y[:, np.newaxis, :], X.shape[1], axis=1)
        flatted_y_new = y_new.reshape(-1, y_new.shape[-1])
        
    flatted_y_new = flatted_y_new[:,classif_obj]

    selected_features = mrmr_classif(X=flatted_X, y=flatted_y_new, K=num_features)

    return selected_features

# f = feature_selector(64, cwt_train_loaded, y_train, 0)
# feature_selections = {
#     'PCA': {
#         'Valence': feature_selector(64, pca_train_loaded, y_train, 0), #1024x40x8064
#         'Arousal': feature_selector(64, pca_train_loaded, y_train, 1) #1024x40x8064
#     },
#     'WT': {
#         'Valence': feature_selector(64, cwt_train_loaded, y_train, 0), #1024x40x128
#         'Arousal': feature_selector(64, cwt_train_loaded, y_train, 1) #1024x40x128
#     }
# }

# prefix_output_path = '/kaggle/working/'
# filename = "feature_selections.pkl"

# with open(filename, 'wb') as file:
#     pickle.dump(feature_selections, file)


In [12]:
# ===============================
# Φόρτωση επιλεγμένων χαρακτηριστικών
# ===============================
prefix_path = '/kaggle/input/feature-selection-deap/'
filename = "feature_selections.pkl"

with open(prefix_path + filename, 'rb') as file:
    feature_selection_loaded = pickle.load(file)

print(feature_selection_loaded)

{'PCA': {'Valence': [62, 96, 53, 87, 15, 66, 83, 11, 113, 46, 125, 39, 38, 116, 6, 68, 56, 57, 36, 123, 126, 120, 69, 10, 51, 9, 119, 35, 99, 94, 43, 40, 42, 52, 90, 59, 16, 105, 118, 64, 27, 49, 63, 124, 101, 17, 98, 86, 5, 111, 81, 80, 33, 109, 45, 8, 61, 54, 1, 50, 112, 37, 29, 121], 'Arousal': [98, 59, 94, 9, 72, 97, 47, 56, 21, 17, 71, 83, 50, 5, 2, 39, 69, 49, 107, 37, 48, 65, 114, 27, 38, 20, 42, 11, 8, 112, 80, 81, 84, 14, 0, 91, 120, 1, 40, 118, 123, 28, 12, 86, 46, 104, 4, 74, 108, 3, 125, 88, 122, 55, 106, 29, 115, 30, 113, 66, 87, 119, 23, 41]}, 'WT': {'Valence': [52, 51, 53, 54, 50, 55, 56, 49, 57, 58, 48, 59, 1, 60, 61, 62, 63, 47, 64, 5, 65, 6, 66, 4, 67, 7, 68, 46, 0, 69, 8, 3, 70, 9, 10, 71, 30, 2, 31, 29, 11, 32, 17, 72, 16, 12, 18, 45, 15, 28, 13, 73, 14, 33, 19, 74, 20, 27, 34, 75, 21, 26, 35, 44], 'Arousal': [64, 63, 65, 66, 62, 67, 61, 68, 60, 69, 70, 59, 71, 58, 72, 73, 33, 57, 32, 34, 74, 31, 75, 56, 35, 76, 30, 36, 77, 55, 78, 37, 29, 79, 54, 80, 38, 81, 28, 53

In [13]:
# Ενημέρωση και εγκατάσταση του synthesizer και του επίσημου SoundFont
!apt-get update
!apt-get install -y fluidsynth fluid-soundfont-gm
!pip install midi2audio librosa

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,225 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,572 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.l

In [14]:
# import librosa
# import numpy as np
# import os
# from midi2audio import FluidSynth
# from sklearn.model_selection import train_test_split

# # Το μονοπάτι αυτό είναι στάνταρ για το Linux του Kaggle μετά την εγκατάσταση
# sf2_path = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

# if not os.path.exists(sf2_path):
#     print("Κάτι πήγε στραβά με την εγκατάσταση. Βεβαιώσου ότι το Internet είναι ON.")
# else:
#     fs = FluidSynth(sf2_path)
#     print("Ο Synthesizer είναι έτοιμος!")

# def extract_advanced_features(midi_file):
#     temp_wav = "temp_audio.wav"
#     try:
#         # 1. Σύνθεση MIDI -> WAV
#         fs.midi_to_audio(midi_file, temp_wav)
        
#         # 2. Φόρτωση ήχου
#         y, sr = librosa.load(temp_wav)
        
#         # 3. Chroma Features (12 τιμές - αντιπροσωπεύουν την αρμονία/κλίμακα)
#         chroma = librosa.feature.chroma_stft(y=y, sr=sr)
#         chroma_mean = np.mean(chroma, axis=1)
        
#         # 4. Spectral Flux / Onset Strength (1 τιμή - αντιπροσωπεύει τον ρυθμό/ενέργεια)
#         onset_env = librosa.onset.onset_strength(y=y, sr=sr)
#         spectral_flux = np.mean(onset_env)
        
#         # 5. Spectral Rolloff (1 τιμή - βοηθά στο διαχωρισμό "φωτεινών" vs "σκοτεινών" ήχων)
#         rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
#         rolloff_mean = np.mean(rolloff)

#         # Καθαρισμός προσωρινού αρχείου
#         if os.path.exists(temp_wav):
#             os.remove(temp_wav)
            
#         return np.hstack(([spectral_flux, rolloff_mean], chroma_mean)) # Σύνολο 14 χαρακτηριστικά
#     except Exception as e:
#         print(f"Error in {midi_file}: {e}")
#         return np.zeros(14)

# # Μαζική επεξεργασία και των 36 αρχείων
# audio_features = []
# midi_dir = '/kaggle/input/deap-dataset/deap-dataset/audio_stimuli_MIDI/'

# for i in range(1, 37):
#     path = os.path.join(midi_dir, f'exp_id_{i}.mid')
#     print(f"Processing MIDI {i}/36...") 
#     audio_features.append(extract_advanced_features(path))

# audio_features_array = np.array(audio_features)

# num_subjects = 32
# num_trials_midi = 36
# num_total_trials = 40

# # Reshape labels από το αρχικό σας subjects['labels']
# labels_reshaped = subjects['labels'].reshape(num_subjects, num_total_trials, 2)

# X_final = []
# y_final = []

# for s in range(num_subjects):
#     for t in range(num_trials_midi):
#         X_final.append(audio_features_array[t]) # Τα χαρακτηριστικά του ήχου
#         y_final.append(labels_reshaped[s, t, :]) # Οι ετικέτες του χρήστη s για το trial t

# X_final = np.array(X_final) # Shape: (1152, 14)
# y_final = np.array(y_final) # Shape: (1152, 2)

# from sklearn.model_selection import train_test_split

# # 1. Split σε Train και Test (80/20)
# X_train_au, X_test_au, y_train_au, y_test_au = train_test_split(
#     X_final, y_final, test_size=0.2, random_state=42
# )

# # 2. Reshape για τα Deep Learning μοντέλα (RNN/CNN) 
# # Από (Samples, Features) σε (Samples, 1, Features)
# X_train_au_dl = X_train_au.reshape(X_train_au.shape[0], 1, X_train_au.shape[1])
# X_test_au_dl = X_test_au.reshape(X_test_au.shape[0], 1, X_test_au.shape[1])

# # 3. Δημιουργία της δομής aggregated_data ειδικά για τον ήχο
# # Θα φτιάξουμε δύο πειράματα: ένα για Arousal και ένα για Valence
# aggregated_audio_experiments = [
#     ['AUDIO_ADV', 'NO', 'AROUSAL', {
#         'TRAIN': (X_train_au, y_train_au), 
#         'TEST': (X_test_au, y_test_au),
#         'DL_TRAIN': (X_train_au_dl, y_train_au), # Για CNN/RNN
#         'DL_TEST': (X_test_au_dl, y_test_au)
#     }],
#     ['AUDIO_ADV', 'NO', 'VALENCE', {
#         'TRAIN': (X_train_au, y_train_au), 
#         'TEST': (X_test_au, y_test_au),
#         'DL_TRAIN': (X_train_au_dl, y_train_au),
#         'DL_TEST': (X_test_au_dl, y_test_au)
#     }]
# ]

In [15]:
# ===============================
# Δημιουργία πειραματικών συνδυασμών
# ===============================

aggregated_data = [
    ['WT', 'YES', 'VALENCE', {'TRAIN': (cwt_train_loaded[:, :, feature_selection_loaded['WT']['Valence']], y_train), 
                              'TEST': (cwt_test_loaded[:, :, feature_selection_loaded['WT']['Valence']], y_test)}],
    ['WT', 'YES', 'AROUSAL', {'TRAIN': (cwt_train_loaded[:, :, feature_selection_loaded['WT']['Arousal']], y_train),
                              'TEST': (cwt_test_loaded[:, :, feature_selection_loaded['WT']['Arousal']], y_test)}],

    ['WT', 'NO', '*', {'TRAIN': (cwt_train_loaded, y_train), 
                       'TEST': (cwt_test_loaded, y_test)}],

    ['PE', 'NO', '*', {'TRAIN': (pe_train_loaded, y_train), 
                       'TEST': (pe_test_loaded, y_test)}],

    ['NONE', 'YES', 'VALENCE', {'TRAIN': (pca_train_loaded[:, :, feature_selection_loaded['PCA']['Valence']], y_train), 
                                'TEST': (pca_test_loaded[:, :, feature_selection_loaded['PCA']['Valence']], y_test)}],
    ['NONE', 'YES', 'AROUSAL', {'TRAIN': (pca_train_loaded[:, :, feature_selection_loaded['PCA']['Arousal']], y_train), 
                                'TEST': (pca_test_loaded[:, :, feature_selection_loaded['PCA']['Arousal']], y_test)}],

    ['NONE', 'NO', '*', {'TRAIN': (pca_train_loaded, y_train), 
                         'TEST': (pca_test_loaded, y_test)}],
]

## Model

In [16]:
# ===============================
# Ορισμός Metrics Αξιολόγησης
# ===============================

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F
from torch.optim import Adam
from tqdm import tqdm
from torch.optim.lr_scheduler import StepLR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def calculate_metric(metric, y_true, y_pred):
    metric = metric.lower()  # Chuyển metric về chữ thường để xử lý dễ dàng hơn

    if metric == 'acc':
        return accuracy_score(y_true, y_pred)
    elif metric == 'sens':  # Sensitivity (Recall)
        return recall_score(y_true, y_pred, average='binary')
    elif metric == 'spec':  # Specificity
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return tn / (tn + fp) if (tn + fp) != 0 else 0.0
    elif metric == 'prec':  # Precision
        return precision_score(y_true, y_pred, average='binary')
    elif metric == 'f-measure':  # F1-Score
        return f1_score(y_true, y_pred, average='binary')
    else:
        raise ValueError(f"Metric '{metric}'  ['Acc', 'Sens', 'Spec', 'Prec', 'F-measure'].")
    

def evaluate(model, name_model, classifi_obj, X, y):

    if name_model in ['svm', 'rf']:
        X = X.reshape(X.shape[0], -1)
        y_pred = model.predict(X)
        
    else:
        X = torch.as_tensor(X, dtype=torch.float32).to(DEVICE)
        if name_model == 'rnn':
            X = X.permute(0, 2, 1)
        y_pred = model(X)
        y_pred = (F.sigmoid(y_pred) >= 0.5).float().cpu().numpy()

    result = {
        metric: calculate_metric(
            metric, 
            y[:, classifi_obj],
            y_pred
        )
        for metric in ['Acc', 'Sens', 'Spec', 'Prec', 'F-measure']
    }

    return result

In [17]:
class EarlyStopping:
    def __init__(self, patience=5, delta=1e-3, save_path='checkpoint.pth'):
        self.patience = patience
        self.delta = delta
        self.save_path = save_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            # self.save_checkpoint(val_loss, model)
        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            # self.save_checkpoint(val_loss, model)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.save_path)
        print(f"Validation loss : {val_loss:.6f}, model .")

def setup_config_train_deep_model(train_data_X, train_data_y, test_data_X, test_data_y, name_model):
    batch_size = 64
    device=DEVICE
    
    # Tạo TensorDataset cho train và test
    train_dataset = TensorDataset(torch.as_tensor(train_data_X, device=device), torch.as_tensor(train_data_y, device=device))
    test_dataset = TensorDataset(torch.as_tensor(test_data_X, device=device), torch.as_tensor(test_data_y, device=device))
    
    # Tạo DataLoader cho train và test với batch_size=64 (hoặc tùy chỉnh)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    num_epochs = 20
    optimizer = Adam
    lr = 1e-4
    prefix_output_path = '/kaggle/working/'

    config = {
        'train_dataloader': train_dataloader,
        'test_dataloader': test_dataloader,
        'num_epochs': num_epochs,
        'lr': lr,
        'optimizer': optimizer,
        'prefix_output_path': prefix_output_path,
        'name_model': name_model
    }

    return config
    
# ===============================
# Εκπαίδευση Deep Learning Μοντέλων
# ===============================

def train_deep_model(model, classifi_obj, **kwargs):
    name_model = kwargs.get('name_model')
    lr = kwargs.get('lr')
    optimizer = kwargs.get('optimizer')(model.parameters(), lr=lr)
    num_epochs = kwargs.get('num_epochs')
    
    scheduler = StepLR(optimizer, step_size=7, gamma=0.9)
    criterion = nn.BCEWithLogitsLoss()
    train_loss = []
    val_loss = []
    log = ""
    
    for epoch in tqdm(range(num_epochs)):
        model.train()
        train_dataloader = kwargs.get('train_dataloader')
        train_epoch_loss = []
        
        for i, (X_batch, y_batch) in enumerate(train_dataloader):
            optimizer.zero_grad()

            X_batch = X_batch.to(torch.float32)
            y_batch = y_batch.to(torch.float32)
            
            if name_model.lower() == 'rnn':
                X_batch = X_batch.permute(0,2,1)
                
            y_pred = model(X_batch)
            
            loss = criterion(y_pred, y_batch[:, classifi_obj])
            train_epoch_loss.append(loss.item())
            
            loss.backward()
            optimizer.step()

        scheduler.step()
        train_epoch_loss = np.mean(train_epoch_loss)
        train_loss.append(train_epoch_loss)

        # Tính validation loss
        test_dataloader = kwargs.get('test_dataloader')
        model.eval()  
        val_epoch_loss = []
        with torch.no_grad():
            for X_batch, y_batch in test_dataloader:
                
                X_batch = X_batch.to(torch.float32)
                y_batch = y_batch.to(torch.float32)

                if name_model.lower() == 'rnn':
                    X_batch = X_batch.permute(0,2,1)
                
                outputs = model(X_batch)
                
                loss = criterion(outputs, y_batch[:, classifi_obj])
                val_epoch_loss.append(loss.item())

        val_epoch_loss = np.mean(val_epoch_loss)
        val_loss.append(val_epoch_loss)

        prefix_output_path = kwargs.get('prefix_output_path')
                
        log += '\n' + f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_epoch_loss:.4f}, Val Loss: {val_epoch_loss:.4f}, lr: {scheduler.get_last_lr()}"
        
        early_stopping = EarlyStopping(val_loss, model, save_path=f'{prefix_output_path}{name_model}.pth')
        if early_stopping.early_stop:
            print("Early stopping!")
            # early_stopping.save_checkpoint(val_loss, model)
            break
            
    # with open(f'{prefix_output_path}log_{name_model}.txt', 'w') as file:
    #     file.write(log)

    return train_loss, val_loss, log
    # pass

def setup_config_train_ml_model(X_train, y_train, X_test, y_test, name_model):
    return {
        'X_train': X_train,
        'y_train': y_train,
        'name_model': name_model.lower()
    }

# ===============================
# Εκπαίδευση ML Μοντέλων
# ===============================

def train_ml_model(model, classifi_obj, **kwargs):
    name_model = kwargs.get('name_model').lower()
    X_train = kwargs.get('X_train')
    X_train = X_train.reshape(X_train.shape[0], -1)
    y = kwargs.get('y_train')[:, classifi_obj]
    model.fit(X_train, y)

def setup_config_train(X_train, y_train, X_test, y_test, name_model):
    name_model = name_model.lower()
    if name_model in ['svm', 'rf']:
        
        return setup_config_train_ml_model(X_train, y_train, X_test, y_test, name_model)

    else:

        return setup_config_train_deep_model(X_train, y_train, X_test, y_test, name_model)

def train_model(model, classifi_obj, **kwargs):
    if kwargs.get('name_model').lower() in ['svm', 'rf']:

        return train_ml_model(model, classifi_obj, **kwargs)

    else:

        return train_deep_model(model, classifi_obj, **kwargs)
    

In [18]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn

class RNNModel(nn.Module):
    """
    LSTM για EEG χρονοσειρές
    """
    def __init__(self, input_size, hidden_size, num_layers, output_size, rnn_type="LSTM", device=DEVICE):
        super(RNNModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Choose RNN type: LSTM, GRU, or vanilla RNN
        if rnn_type == "LSTM":
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, device=device)
        elif rnn_type == "GRU":
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, device=device)
        else:
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, device=device)

        # Fully connected layer for output
        self.fc = nn.Linear(hidden_size, output_size, device=device)

    def forward(self, x):
        # Initialize hidden state and cell state (if using LSTM)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # Hidden state

        if isinstance(self.rnn, nn.LSTM):
            c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # Cell state
            out, _ = self.rnn(x, (h0, c0))
        else:
            out, _ = self.rnn(x, h0)

        # Take the last output of the sequence
        out = self.fc(out[:, -1, :])

        out = out.squeeze()
        
        return out

class CNNModel(nn.Module):
    """
    CNN 1D για EEG δεδομένα
    """
    def __init__(self, in_channels, outs_channels, output_size, length_signal, device=DEVICE):
        super().__init__()

        channels = [in_channels] + outs_channels
        self.model = nn.Sequential(
            *[nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, device=device),
                nn.BatchNorm1d(out_channels, device=device),
                nn.ReLU()
            ) for in_channels, out_channels in zip(channels[:-1], channels[1:])]
        )
            
        self.fc = nn.Linear(outs_channels[-1] * length_signal, output_size, device=device) 

    def forward(self, x):
        x = self.model(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        x = x.squeeze()

        return x
    
def get_model(**kwargs):
    model = None
    name_model = kwargs.get('name_model').lower()
    
    device = kwargs.get('device', DEVICE)
    
    if name_model == 'svm':
        model = SVC(kernel='rbf')
        
    elif name_model == 'rf':
        model = RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=5, random_state=42)
        
    elif name_model == 'rnn':
        # with torch.no_grad():
        #     dummy = torch.Tensor(1024, 40, 128).permute(0, 2, 1)# -> 1024, 128, 40
        input_features = kwargs.get('input_features', 40)
        hidden_size = kwargs.get('hidden_size', 128)
        num_layers = kwargs.get('num_layers', 4)
        output_size = kwargs.get('output_size', 1)
        length_signal = kwargs.get('length_signal', 128)
        
        model = RNNModel(
            input_size=input_features, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            output_size=output_size, 
            rnn_type="LSTM"
        )
        
    elif name_model == 'cnn':
        in_channels = kwargs.get('in_channels', 40)
        outs_channels = kwargs.get('outs_channels', [64,256,128])
        output_size = kwargs.get('output_size', 1)
        length_signal = kwargs.get('length_signal', 128)
        
        model = CNNModel(
            in_channels=in_channels,
            outs_channels=outs_channels,
            output_size=output_size,
            length_signal=length_signal,
            device=device
        )

    return model

In [19]:
# X_data_train, y_data_train = aggregated_data[0][-1]['TRAIN']
# X_data_test, y_data_test = aggregated_data[0][-1]['TEST']

# name_model = 'svm'
# classifi_obj = 0

# config = {'name_model': name_model, 'length_signal': 64}
# model = get_model(**config)

# config_train = setup_config_train(X_data_train, y_data_train, X_data_test, y_data_test, name_model)
# # print(config_train)
# info = train_model(model, classifi_obj, **config_train)
# print(info)

# print(evaluate(model, name_model, classifi_obj, X_data_test, y_data_test))
# # out = model(X)
# # print(out.shape)

In [20]:
# results_audio = []

# for name_model in ['rf', 'svm', 'rnn', 'cnn']:
#     print(f"\n--- Training Model: {name_model.upper()} with Audio Features ---")
    
#     for experiment in aggregated_audio_experiments:
#         feat_name, fs_status, target, data_dict = experiment
        
#         # Επιλογή σωστών δεδομένων ανάλογα με το μοντέλο
#         if name_model in ['rnn', 'cnn']:
#             X_tr, y_tr = data_dict['DL_TRAIN']
#             X_ts, y_ts = data_dict['DL_TEST']
#         else:
#             X_tr, y_tr = data_dict['TRAIN']
#             X_ts, y_ts = data_dict['TEST']
            
#         # Καθορισμός στόχου (Valence=0, Arousal=1 βάσει του κώδικά σου)
#         classifi_obj = 1 if target == 'AROUSAL' else 0
        
#         # 1. Δημιουργία μοντέλου
#         config = {
#             'length_signal': X_tr.shape[-1], # 14 χαρακτηριστικά
#             'name_model': name_model
#         }
#         model = get_model(**config)
        
#         # 2. Προπόνηση
#         config_train = setup_config_train(X_tr, y_tr, X_ts, y_ts, name_model)
#         train_model(model, classifi_obj, **config_train)
        
#         # 3. Αξιολόγηση
#         acc = evaluate(model, name_model, classifi_obj, X_ts, y_ts)
        
#         print(f"Result: {name_model} | {target} | Accuracy: {acc}%")
#         results_audio.append((name_model, target, acc))

# # Εκτύπωση τελικής σύνοψης
# print("\n=== FINAL AUDIO RESULTS ===")
# for res in results_audio:
#     print(res)

In [21]:
# ===============================
# Εκτέλεση Πειραμάτων
# ===============================

models = []
evaluates = []
i = 0
for name_model in ['rf', 'svm', 'rnn', 'cnn']:
    for experiment in aggregated_data:
        data = experiment[-1]
        classifi_objs = [0] if experiment[2] == 'VALENCE' else [1] if experiment[2] == 'AROUSAL' else [0,1]
        (X_train_data, y_train_data), (X_test_data, y_test_data) = data['TRAIN'], data['TEST']
        # print(X_test_data.shape, X_train_data.shape)
        for classifi_obj in classifi_objs:
            config = {
                'length_signal': X_train_data.shape[-1],
                'name_model': name_model
            }
            
            model = get_model(**config)

            # print(name_model)
            config_train = setup_config_train(X_train_data, y_train_data, X_test_data, y_test_data, name_model)
            train_model(model, classifi_obj, **config_train)

            motion = 'VALENCE' if classifi_obj else 'AROUSAL'

            models.append((*experiment[:2], motion, model))

            metrics = evaluate(model, name_model, classifi_obj, X_test_data, y_test_data)

            info = (
                name_model,
                *experiment[:2],
                motion,
                {
                    'Acc': metrics['Acc'],
                    'Sens': metrics['Sens'],
                    'Spec': metrics['Spec'],
                    'Prec': metrics['Prec'],
                    'F1': metrics['F-measure']
                }
            )
            
            i += 1
            print(f"experiment {i}: {info}")
            evaluates.append(info)

            
            

experiment 1: ('rf', 'WT', 'YES', 'AROUSAL', {'Acc': 0.78125, 'Sens': 0.07547169811320754, 'Spec': 0.9655172413793104, 'Prec': 0.36363636363636365, 'F1': 0.125})
experiment 2: ('rf', 'WT', 'YES', 'VALENCE', {'Acc': 0.76953125, 'Sens': 0.08620689655172414, 'Spec': 0.9696969696969697, 'Prec': 0.45454545454545453, 'F1': 0.14492753623188406})
experiment 3: ('rf', 'WT', 'NO', 'AROUSAL', {'Acc': 0.78125, 'Sens': 0.07547169811320754, 'Spec': 0.9655172413793104, 'Prec': 0.36363636363636365, 'F1': 0.125})
experiment 4: ('rf', 'WT', 'NO', 'VALENCE', {'Acc': 0.765625, 'Sens': 0.06896551724137931, 'Spec': 0.9696969696969697, 'Prec': 0.4, 'F1': 0.1176470588235294})
experiment 5: ('rf', 'PE', 'NO', 'AROUSAL', {'Acc': 0.7890625, 'Sens': 0.0, 'Spec': 0.9950738916256158, 'Prec': 0.0, 'F1': 0.0})
experiment 6: ('rf', 'PE', 'NO', 'VALENCE', {'Acc': 0.7734375, 'Sens': 0.034482758620689655, 'Spec': 0.98989898989899, 'Prec': 0.5, 'F1': 0.06451612903225806})
experiment 7: ('rf', 'NONE', 'YES', 'AROUSAL', {'A

100%|██████████| 20/20 [01:30<00:00,  4.50s/it]


experiment 21: ('rnn', 'WT', 'YES', 'AROUSAL', {'Acc': 0.79296875, 'Sens': 0.0, 'Spec': 1.0, 'Prec': 0.0, 'F1': 0.0})


100%|██████████| 20/20 [01:30<00:00,  4.51s/it]


experiment 22: ('rnn', 'WT', 'YES', 'VALENCE', {'Acc': 0.76953125, 'Sens': 0.05172413793103448, 'Spec': 0.9797979797979798, 'Prec': 0.42857142857142855, 'F1': 0.0923076923076923})


...

experiment 38: ('cnn', 'NONE', 'YES', 'VALENCE', {'Acc': 0.75390625, 'Sens': 0.034482758620689655, 'Spec': 0.9646464646464646, 'Prec': 0.2222222222222222, 'F1': 0.05970149253731343})


100%|██████████| 20/20 [00:37<00:00,  1.86s/it]


experiment 39: ('cnn', 'NONE', 'NO', 'AROUSAL', {'Acc': 0.78515625, 'Sens': 0.018867924528301886, 'Spec': 0.9852216748768473, 'Prec': 0.25, 'F1': 0.03508771929824561})


100%|██████████| 20/20 [00:35<00:00,  1.79s/it]


experiment 40: ('cnn', 'NONE', 'NO', 'VALENCE', {'Acc': 0.74609375, 'Sens': 0.05172413793103448, 'Spec': 0.9494949494949495, 'Prec': 0.23076923076923078, 'F1': 0.08450704225352113})
